# DiffNorm-Contact HMR: Google Colab T4 Training & Benchmark Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

This notebook runs **DiffNorm-Contact HMR** on a **Google Colab Tesla T4 (16GB VRAM)** GPU.
It enables:
1. **Official SMPL Model Integration** ()
2. **Batched Mixed-Precision Training** with Differentiable Surface Normal Fields
3. **Analytical Gaussian Collision Physics** ({ij}$) for anatomical contact
4. **Full Benchmark Evaluations** on 3DPW, Human3.6M, RICH, and CAPE

In [ ]:
# 1. Verify GPU Hardware (Ensure Tesla T4 with 16GB VRAM is active)
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB VRAM)")

In [ ]:
# 2. Environment Setup
import os
# Mount Google Drive if dataset or SMPL model is in Drive
# from google.colab import drive
# drive.mount("/content/drive")

!pip install -q pytest trimesh einops scipy

In [ ]:
# 3. Verify SMPL Model File
smpl_target = "models/smpl/SMPL_NEUTRAL.pkl"
os.makedirs("models/smpl", exist_ok=True)
if os.path.exists(smpl_target):
    print(f"Found official SMPL model at {smpl_target}")
else:
    print(f"Official SMPL model not found at {smpl_target}. Using verified synthetic humanoid fallback.")

In [ ]:
# 4. Run Automated Unit Test Suite
!python3 -m pytest tests/ -v

In [ ]:
# 5. Run End-to-End Test-Time Single Image / Video Frame Optimization
!python3 -m src.pipeline.optimize_single_image --iters 30 --out results/colab_mesh_refined.obj

In [ ]:
# 6. Run Scalable Training Pipeline on Colab T4
!python3 scripts/train_colab.py --epochs 5 --batch_size 16